# Exercise 3 — Heterogeneous treatment effects: R-learner, DR-learner, and causal forests

**Estimated time:** 30 minutes  
**Lecture notation:** CATE $\tau(w)=\mathbb E[Y^1-Y^0\mid W=w]$, outcome regression $Q(w,a)$, marginal outcome regression $\mu(w)=\mathbb E[Y\mid W=w]$, and propensity score $\varpi(w)=\pi(1\mid w)$.

## Learning goals

1. Estimate $\tau(w)$ with an R-learner.
2. Estimate $\tau(w)$ with a DR-learner.
3. Compare them to a causal forest implementation when available.
4. Diagnose conceptual mistakes that still produce runnable code, especially using the wrong residualization or pseudo-outcome.

## Colab instructions

Use **File → Save a copy in Drive** before editing. You can also download the notebook with **File → Download → Download .ipynb**.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sys, subprocess, importlib.util
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

np.random.seed(456)
plt.rcParams["figure.figsize"] = (8, 4.5)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


## Optional package for causal forests

The R-learner and DR-learner use only `scikit-learn`. For a real causal forest, this notebook tries to use `econml`. If installation is slow or unavailable, the notebook falls back to a lightweight forest-style local R-learner so the lab can continue.


In [ ]:
INSTALL_ECONML_IF_MISSING = True  # Set to False if installation is slow and you want to use the fallback.

if importlib.util.find_spec("econml") is None and INSTALL_ECONML_IF_MISSING:
    print("Installing econml. In Colab this may take a couple of minutes.")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "econml"])
    except Exception as err:
        print("Could not install econml automatically. The notebook will use the fallback.")
        print("Reason:", repr(err))

try:
    from econml.dml import CausalForestDML
    ECONML_AVAILABLE = True
    print("econml is available: CausalForestDML will be used.")
except Exception as err:
    ECONML_AVAILABLE = False
    print("econml is not available or failed to import. A fast fallback will be used.")
    print("Reason:", repr(err))


## 1. HTE simulation

The direct treatment effect is heterogeneous:

$$
\tau_{\rm direct}(W)=1+1.2\mathbb I(W_1>0)-0.7W_2+0.5W_1W_3.
$$

The post-treatment variable `M` is intentionally included as a trap. It lies on a path from $A$ to $Y$, so the total CATE stored as `tau_true` also includes the mediated component. It should not be used as a baseline effect modifier for the total CATE.


In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def simulate_hte(n=3500, seed=1, exercise=False):
    rng = np.random.default_rng(seed)
    W1 = rng.normal(size=n)
    W2 = rng.normal(size=n)
    W3 = rng.normal(size=n)
    W4 = rng.normal(size=n) if exercise else np.zeros(n)

    lin_pi = -0.15 + 0.7 * W1 - 0.6 * W2 + 0.35 * W3 + (0.45 * W4 if exercise else 0)
    pi = np.clip(sigmoid(lin_pi), 0.05, 0.95)
    A = rng.binomial(1, pi)

    tau = 1.0 + 1.2 * (W1 > 0).astype(float) - 0.7 * W2 + 0.5 * W1 * W3 + (0.4 * W4 if exercise else 0)
    mu0 = 1.0 + np.sin(W1) + 0.5 * W2**2 - 0.4 * W3 + (0.25 * W4 if exercise else 0)
    M = 0.8 * A + 0.6 * W1 - 0.2 * W2 + rng.normal(scale=0.7, size=n)
    Y = mu0 + A * tau + 0.3 * M + rng.normal(scale=1.0, size=n)

    total_tau = tau + 0.3 * 0.8  # total effect includes the mediated component A -> M -> Y
    df = pd.DataFrame({"W1": W1, "W2": W2, "W3": W3, "W4": W4, "A": A, "M": M, "Y": Y, "pi_true": pi, "tau_true": total_tau, "tau_direct": tau})
    return df

df = simulate_hte(n=2500, seed=11, exercise=False)
W_cols = ["W1", "W2", "W3"]
print(df.head())
print(f"True ATE E[tau(W)] = {df['tau_true'].mean():.3f}")


## 2. Cross-fitted nuisance functions

For the R-learner, we need

$$
\mu(w)=\mathbb E[Y\mid W=w],\qquad \varpi(w)=\mathbb P(A=1\mid W=w).
$$

For the DR-learner, we need $Q(w,0)$, $Q(w,1)$, and $\varpi(w)$.


In [ ]:
def crossfit_hte_nuisances(df, W_cols, n_splits=3, seed=123):
    n = len(df)
    mu_hat = np.zeros(n)
    pi_hat = np.zeros(n)
    Q0_hat = np.zeros(n)
    Q1_hat = np.zeros(n)

    mu_model = RandomForestRegressor(n_estimators=100, min_samples_leaf=30, random_state=seed)
    q_model = RandomForestRegressor(n_estimators=100, min_samples_leaf=30, random_state=seed + 1)
    pi_model = RandomForestClassifier(n_estimators=100, min_samples_leaf=30, random_state=seed + 2)

    X = df[W_cols]
    XA = df[W_cols + ["A"]]
    Y = df["Y"].values
    A = df["A"].values

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for train_idx, test_idx in kf.split(df):
        mu = clone(mu_model)
        q = clone(q_model)
        p = clone(pi_model)

        mu.fit(X.iloc[train_idx], Y[train_idx])
        q.fit(XA.iloc[train_idx], Y[train_idx])
        p.fit(X.iloc[train_idx], A[train_idx])

        X_test = X.iloc[test_idx].copy()
        X1 = X_test.copy(); X1["A"] = 1
        X0 = X_test.copy(); X0["A"] = 0

        mu_hat[test_idx] = mu.predict(X_test)
        Q1_hat[test_idx] = q.predict(X1[W_cols + ["A"]])
        Q0_hat[test_idx] = q.predict(X0[W_cols + ["A"]])
        pi_hat[test_idx] = p.predict_proba(X_test)[:, 1]

    return mu_hat, np.clip(pi_hat, 0.03, 0.97), Q0_hat, Q1_hat

mu_hat, pi_hat, Q0_hat, Q1_hat = crossfit_hte_nuisances(df, W_cols, n_splits=3)
print(pd.DataFrame({"mu_hat": mu_hat, "pi_hat": pi_hat, "Q0_hat": Q0_hat, "Q1_hat": Q1_hat}).describe())


## 3. Completed R-learner

The R-learner residualizes both outcome and treatment:

$$
Y-\widehat\mu(W) \approx \{A-\widehat\varpi(W)\}\tau(W).
$$

Equivalently, regress

$$
\widetilde Y_i = \frac{Y_i-\widehat\mu(W_i)}{A_i-\widehat\varpi(W_i)}
$$

on $W_i$ with weights $\widehat\omega_i=\{A_i-\widehat\varpi(W_i)\}^2$.


In [ ]:
def fit_r_learner(df, W_cols, mu_hat, pi_hat, seed=123):
    A = df["A"].values
    Y = df["Y"].values
    denom = A - pi_hat
    denom = np.where(np.abs(denom) < 1e-3, np.sign(denom) * 1e-3, denom)

    pseudo_y = (Y - mu_hat) / denom
    weights = (A - pi_hat) ** 2

    tau_model = RandomForestRegressor(n_estimators=150, min_samples_leaf=30, random_state=seed)
    tau_model.fit(df[W_cols], pseudo_y, sample_weight=weights)
    tau_hat = tau_model.predict(df[W_cols])
    return tau_hat, tau_model, pseudo_y, weights

tau_r, r_model, pseudo_r, weights_r = fit_r_learner(df, W_cols, mu_hat, pi_hat)
print(f"R-learner estimated ATE by averaging tau_hat(W): {tau_r.mean():.3f}")
print(f"R-learner RMSE against true tau(W): {rmse(df['tau_true'], tau_r):.3f}")


## 4. Completed DR-learner

The DR-learner creates a doubly robust pseudo-outcome for the CATE:

$$
\widehat\varphi_i = \widehat Q(W_i,1)-\widehat Q(W_i,0)+
\left\{\frac{A_i}{\widehat\varpi(W_i)}-\frac{1-A_i}{1-\widehat\varpi(W_i)}\right\}
\{Y_i-\widehat Q(W_i,A_i)\}.
$$

Then regress $\widehat\varphi_i$ on $W_i$.


In [ ]:
def fit_dr_learner(df, W_cols, Q0_hat, Q1_hat, pi_hat, seed=456):
    A = df["A"].values
    Y = df["Y"].values
    Q_A_hat = np.where(A == 1, Q1_hat, Q0_hat)
    phi = (Q1_hat - Q0_hat) + (A / pi_hat - (1 - A) / (1 - pi_hat)) * (Y - Q_A_hat)

    tau_model = RandomForestRegressor(n_estimators=150, min_samples_leaf=30, random_state=seed)
    tau_model.fit(df[W_cols], phi)
    tau_hat = tau_model.predict(df[W_cols])
    return tau_hat, tau_model, phi

tau_dr, dr_model, pseudo_dr = fit_dr_learner(df, W_cols, Q0_hat, Q1_hat, pi_hat)
print(f"DR-learner estimated ATE by averaging tau_hat(W): {tau_dr.mean():.3f}")
print(f"DR-learner RMSE against true tau(W): {rmse(df['tau_true'], tau_dr):.3f}")


## 5. Causal forest

When `econml` is available, we use `CausalForestDML`. This implements an orthogonal forest approach: nuisance functions are estimated and local treatment effects are learned through forest weights.

If `econml` is not available, the fallback below uses the R-learner prediction as a forest-style proxy so that the visualization cells still run. The fallback is not a replacement for a production causal forest.


In [ ]:
def fit_causal_forest_or_fallback(df, W_cols, seed=789):
    X = df[W_cols].values
    Y = df["Y"].values
    A = df["A"].values

    if ECONML_AVAILABLE:
        cf = CausalForestDML(
            model_y=RandomForestRegressor(n_estimators=80, min_samples_leaf=30, random_state=seed),
            model_t=RandomForestClassifier(n_estimators=80, min_samples_leaf=30, random_state=seed + 1),
            discrete_treatment=True,
            n_estimators=100,
            min_samples_leaf=30,
            random_state=seed,
        )
        cf.fit(Y, A, X=X)
        tau_hat = cf.effect(X)
        return tau_hat, cf, "econml CausalForestDML"
    else:
        # Fast fallback: refit a local R-learner forest proxy on the supplied data and W columns.
        # This keeps the lab runnable when econml is unavailable, but it is not a replacement
        # for a production causal forest.
        mu_fb, pi_fb, _, _ = crossfit_hte_nuisances(df, W_cols, n_splits=3, seed=seed)
        tau_hat, model, _, _ = fit_r_learner(df, W_cols, mu_fb, pi_fb, seed=seed)
        return tau_hat, model, "fallback: R-learner forest proxy"

tau_cf, cf_model, cf_label = fit_causal_forest_or_fallback(df, W_cols)
print(cf_label)
print(f"Causal forest estimated ATE by averaging tau_hat(W): {tau_cf.mean():.3f}")
print(f"Causal forest RMSE against true tau(W): {rmse(df['tau_true'], tau_cf):.3f}")


## 6. Visualize HTE estimates

A good HTE method should recover the **shape** of $\tau(w)$, not only the average.


In [ ]:
def hte_summary(df, estimates):
    rows = []
    true_tau = df["tau_true"].values
    for name, tau_hat in estimates.items():
        rows.append({
            "method": name,
            "ATE by average tau_hat": np.mean(tau_hat),
            "RMSE tau_hat": rmse(true_tau, tau_hat),
            "corr(tau_hat, tau_true)": np.corrcoef(true_tau, tau_hat)[0, 1],
        })
    return pd.DataFrame(rows)

estimates = {"R-learner": tau_r, "DR-learner": tau_dr, "causal forest": tau_cf}
hte_summary(df, estimates)


In [ ]:
fig, ax = plt.subplots()
ax.scatter(df["tau_true"], tau_r, alpha=0.35, label="R-learner")
ax.scatter(df["tau_true"], tau_dr, alpha=0.35, label="DR-learner")
ax.scatter(df["tau_true"], tau_cf, alpha=0.35, label="causal forest")
lo = min(df["tau_true"].min(), tau_r.min(), tau_dr.min(), tau_cf.min())
hi = max(df["tau_true"].max(), tau_r.max(), tau_dr.max(), tau_cf.max())
ax.plot([lo, hi], [lo, hi], linestyle="--")
ax.set_xlabel("true tau(W)")
ax.set_ylabel("estimated tau_hat(W)")
ax.set_title("HTE recovery: estimated vs true CATE")
ax.legend()
plt.show()


In [ ]:
# Calibration plot by bins of W1: does the method recover one visible source of heterogeneity?
plot_df = df.copy()
plot_df["tau_r"] = tau_r
plot_df["tau_dr"] = tau_dr
plot_df["tau_cf"] = tau_cf
plot_df["W1_bin"] = pd.qcut(plot_df["W1"], q=10, duplicates="drop")

bin_summary = plot_df.groupby("W1_bin", observed=False).agg(
    W1_mid=("W1", "mean"),
    tau_true=("tau_true", "mean"),
    tau_r=("tau_r", "mean"),
    tau_dr=("tau_dr", "mean"),
    tau_cf=("tau_cf", "mean"),
).reset_index()

fig, ax = plt.subplots()
ax.plot(bin_summary["W1_mid"], bin_summary["tau_true"], marker="o", label="true tau")
ax.plot(bin_summary["W1_mid"], bin_summary["tau_r"], marker="o", label="R-learner")
ax.plot(bin_summary["W1_mid"], bin_summary["tau_dr"], marker="o", label="DR-learner")
ax.plot(bin_summary["W1_mid"], bin_summary["tau_cf"], marker="o", label="causal forest")
ax.set_xlabel("W1 bin mean")
ax.set_ylabel("average tau in bin")
ax.set_title("Does the method recover heterogeneity over W1?")
ax.legend()
plt.show()


# Student task: implement HTE learners on new data

The new data adds `W4` as a baseline confounder and effect modifier. The variable `M` is still post-treatment.

Your code should run for many choices, including wrong ones. The point is to diagnose the estimand and the pseudo-outcome, not to debug Python.


In [ ]:
df_ex = simulate_hte(n=3000, seed=2026, exercise=True)
print(df_ex.head())
print(f"True ATE E[tau(W)] = {df_ex['tau_true'].mean():.3f}")


## Task 1 — Choose the effect-modifier/confounder set $W$

Try at least two choices:

1. a baseline-only choice;
2. a prediction-driven choice that includes `M`.

Then compare what happens.


In [ ]:
# TODO: choose variables to use as W.
# Candidate variables: "W1", "W2", "W3", "W4", "M".
W_cols_ex = ["W1", "W2", "W3", "W4"]  # <-- edit this

mu_ex, pi_ex, Q0_ex, Q1_ex = crossfit_hte_nuisances(df_ex, W_cols_ex, n_splits=3, seed=2026)
print("pi range:", pi_ex.min(), pi_ex.max())


## Task 2 — Complete the R-learner pseudo-regression

Conceptual traps:

- The outcome residual should be $Y-\widehat\mu(W)$, not $Y-\widehat Q(W,A)$.
- The treatment residual should be $A-\widehat\varpi(W)$.
- The final regression is of the pseudo-outcome on $W$, not on treatment $A$.


In [ ]:
def r_learner_student(df, W_cols, mu_hat, pi_hat, seed=111):
    A = df["A"].values
    Y = df["Y"].values

    # TODO 1: residualize outcome and treatment.
    y_res = Y - mu_hat          # <-- edit if you want to test alternatives
    a_res = A - pi_hat          # <-- edit if needed

    denom = np.where(np.abs(a_res) < 1e-3, np.sign(a_res) * 1e-3, a_res)

    # TODO 2: pseudo-outcome and weights for the R-learner.
    pseudo_y = y_res / denom    # <-- edit if needed
    weights = a_res ** 2        # <-- edit if needed

    # TODO 3: regress pseudo_y on W only. Do not include A as a feature for tau(W).
    tau_features = df[W_cols]   # <-- edit only if you want to test the trap

    model = RandomForestRegressor(n_estimators=150, min_samples_leaf=30, random_state=seed)
    model.fit(tau_features, pseudo_y, sample_weight=weights)
    tau_hat = model.predict(tau_features)
    return tau_hat, model, pseudo_y, weights

tau_r_ex, r_model_ex, pseudo_r_ex, weights_r_ex = r_learner_student(df_ex, W_cols_ex, mu_ex, pi_ex)
print(f"Your R-learner ATE by averaging tau_hat(W): {tau_r_ex.mean():.3f}")
print(f"RMSE against true tau(W): {rmse(df_ex['tau_true'], tau_r_ex):.3f}")


## Task 3 — Complete the DR-learner pseudo-outcome

Conceptual traps:

- The correction must use $Q(W_i,A_i)$, not only $Q(W_i,1)$ or only $Q(W_i,0)$.
- The control residual has a negative sign.
- The final regression is of the pseudo-outcome on $W$.


In [ ]:
def dr_learner_student(df, W_cols, Q0_hat, Q1_hat, pi_hat, seed=222):
    A = df["A"].values
    Y = df["Y"].values

    # TODO 1: observed-treatment prediction Q(W_i,A_i).
    Q_A_hat = np.where(A == 1, Q1_hat, Q0_hat)  # <-- edit if needed

    # TODO 2: DR pseudo-outcome for tau(W).
    phi = (Q1_hat - Q0_hat) + (A / pi_hat - (1 - A) / (1 - pi_hat)) * (Y - Q_A_hat)  # <-- edit if needed

    # TODO 3: regress phi on W, not on A.
    tau_features = df[W_cols]  # <-- edit only if you want to test the trap

    model = RandomForestRegressor(n_estimators=150, min_samples_leaf=30, random_state=seed)
    model.fit(tau_features, phi)
    tau_hat = model.predict(tau_features)
    return tau_hat, model, phi

tau_dr_ex, dr_model_ex, pseudo_dr_ex = dr_learner_student(df_ex, W_cols_ex, Q0_ex, Q1_ex, pi_ex)
print(f"Your DR-learner ATE by averaging tau_hat(W): {tau_dr_ex.mean():.3f}")
print(f"RMSE against true tau(W): {rmse(df_ex['tau_true'], tau_dr_ex):.3f}")


## Task 4 — Fit a causal forest and compare

The code is supplied, but your choice of `W_cols_ex` still matters.


In [ ]:
tau_cf_ex, cf_model_ex, cf_label_ex = fit_causal_forest_or_fallback(df_ex, W_cols_ex, seed=2026)
print(cf_label_ex)
print(f"Causal forest ATE by averaging tau_hat(W): {tau_cf_ex.mean():.3f}")
print(f"RMSE against true tau(W): {rmse(df_ex['tau_true'], tau_cf_ex):.3f}")


In [ ]:
student_estimates = {"R-learner": tau_r_ex, "DR-learner": tau_dr_ex, "causal forest": tau_cf_ex}
hte_summary(df_ex, student_estimates)


In [ ]:
fig, ax = plt.subplots()
ax.scatter(df_ex["tau_true"], tau_r_ex, alpha=0.3, label="R-learner")
ax.scatter(df_ex["tau_true"], tau_dr_ex, alpha=0.3, label="DR-learner")
ax.scatter(df_ex["tau_true"], tau_cf_ex, alpha=0.3, label="causal forest")
lo = min(df_ex["tau_true"].min(), tau_r_ex.min(), tau_dr_ex.min(), tau_cf_ex.min())
hi = max(df_ex["tau_true"].max(), tau_r_ex.max(), tau_dr_ex.max(), tau_cf_ex.max())
ax.plot([lo, hi], [lo, hi], linestyle="--")
ax.set_xlabel("true tau(W)")
ax.set_ylabel("estimated tau_hat(W)")
ax.set_title("Student task: estimated vs true CATE")
ax.legend()
plt.show()


In [ ]:
plot_df = df_ex.copy()
plot_df["tau_r"] = tau_r_ex
plot_df["tau_dr"] = tau_dr_ex
plot_df["tau_cf"] = tau_cf_ex
plot_df["W1_bin"] = pd.qcut(plot_df["W1"], q=10, duplicates="drop")

bin_summary = plot_df.groupby("W1_bin", observed=False).agg(
    W1_mid=("W1", "mean"),
    tau_true=("tau_true", "mean"),
    tau_r=("tau_r", "mean"),
    tau_dr=("tau_dr", "mean"),
    tau_cf=("tau_cf", "mean"),
).reset_index()

fig, ax = plt.subplots()
ax.plot(bin_summary["W1_mid"], bin_summary["tau_true"], marker="o", label="true tau")
ax.plot(bin_summary["W1_mid"], bin_summary["tau_r"], marker="o", label="R-learner")
ax.plot(bin_summary["W1_mid"], bin_summary["tau_dr"], marker="o", label="DR-learner")
ax.plot(bin_summary["W1_mid"], bin_summary["tau_cf"], marker="o", label="causal forest")
ax.set_xlabel("W1 bin mean")
ax.set_ylabel("average tau in bin")
ax.set_title("Student task: heterogeneity over W1")
ax.legend()
plt.show()


## Questions for discussion

1. What goes wrong if the final R-learner regression uses `A` as a feature?
2. Why is the R-learner orthogonal but not doubly robust in the AIPW sense?
3. Why can the DR-learner pseudo-outcome be noisy under poor overlap?
4. What changes when you include `M` as an effect modifier?
5. Which method best recovers the shape of $\tau(w)$, not only the average?
